# RQ1 Stage 1 on Kaggle

This notebook runs only Stage 1: the 24-run loss, sampling, and augmentation screen.

## Before you run

1. Open **Settings**.
2. Set **Accelerator** to **GPU P100** or **GPU T4 x2**.
3. Set **Internet** to **On**.
4. Attach the dataset `kmader/skin-cancer-mnist-ham10000` if it is not already attached.

Stage 1 takes about 3 to 4 GPU hours. The notebook writes `stage1_results.zip` at the end.


In [ ]:
import os
import sys

import torch

print("Python:", sys.version.split()[0])
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Working directory:", os.getcwd())


In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/conorkz/melanoma-xai-bias.git"
REPO_DIR = "/kaggle/working/melanoma-xai-bias"

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "timm", "albumentations", "captum", "pyyaml"],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"],
    check=True,
)
print("Repo ready:", os.getcwd())


## Data setup

The next cell links the attached Kaggle HAM10000 dataset if it is present. Otherwise it downloads the original archive from Harvard Dataverse and verifies the MD5 checksums. The download is about 2.8 GB.


In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

os.chdir(REPO_DIR)
images_dir = Path("data/HAM10000/images")
metadata_csv = Path("data/raw/HAM10000_metadata.csv")
images_dir.mkdir(parents=True, exist_ok=True)
metadata_csv.parent.mkdir(parents=True, exist_ok=True)

kaggle_root = Path("/kaggle/input/skin-cancer-mnist-ham10000")
linked = False
if (kaggle_root / "HAM10000_metadata.csv").exists():
    shutil.copy2(kaggle_root / "HAM10000_metadata.csv", metadata_csv)
    for part in ("HAM10000_images_part_1", "HAM10000_images_part_2"):
        part_dir = kaggle_root / part
        if part_dir.exists():
            for source in part_dir.glob("*.jpg"):
                target = images_dir / source.name
                if not target.exists():
                    target.symlink_to(source)
    linked = True
    print("Linked HAM10000 from the attached Kaggle dataset")

n_images = len(list(images_dir.glob("*.jpg")))
n_rows = sum(1 for _ in metadata_csv.open()) - 1 if metadata_csv.exists() else 0
if n_images != 10015 or n_rows != 10015:
    if linked:
        for path in images_dir.glob("*.jpg"):
            path.unlink()
    print("Attached dataset incomplete or absent; downloading from Harvard Dataverse")
    subprocess.run([sys.executable, "scripts/download_data.py"], check=True)
    n_images = len(list(images_dir.glob("*.jpg")))
    n_rows = sum(1 for _ in metadata_csv.open()) - 1

print("Metadata rows:", n_rows)
print("Images:", n_images)
assert n_images == 10015 and n_rows == 10015, "HAM10000 is incomplete"
print("Data ready.")


## Stage 1: loss, sampling, and augmentation screen

24 runs: 8 configurations x 3 seeds, EfficientNet-B0 at 160 px, 15 epochs.

If the session resets, rerun this cell. `--skip-existing` skips completed runs.


In [ ]:
import os
import subprocess
import sys

os.chdir(REPO_DIR)
subprocess.run(
    [
        sys.executable,
        "scripts/run_sweep.py",
        "--config",
        "configs/sweep_stage1.yaml",
        "--skip-existing",
    ],
    check=True,
)


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

os.chdir(REPO_DIR)
subprocess.run(
    [
        sys.executable,
        "scripts/collect_results.py",
        "--runs-dir",
        "outputs/runs",
        "--run-prefix",
        "stage1",
        "--out-dir",
        "results",
        "--prefix",
        "rq1_stage1",
    ],
    check=True,
)

display(Markdown(Path("results/rq1_stage1.md").read_text()))
table = pd.read_csv("results/rq1_stage1.csv")
selection_columns = [
    "config_id",
    "n_seeds",
    "val_sensitivity_mean",
    "val_sensitivity_std",
    "val_balanced_accuracy_mean",
    "sensitivity_mean",
    "sensitivity_std",
    "sensitivity_ci_low",
    "sensitivity_ci_high",
]
display(table.sort_values("val_sensitivity_mean", ascending=False)[selection_columns])


In [ ]:
import os
import subprocess
from pathlib import Path

os.chdir(REPO_DIR)
subprocess.run(
    ["zip", "-r", "-q", "stage1_results.zip", "results", "outputs/runs"],
    check=True,
)
print("Download this file from the Kaggle Output pane:")
print(Path("stage1_results.zip").resolve())


## Next steps

1. Download `stage1_results.zip`.
2. Open `results/rq1_stage1.csv` and choose the winner by `val_sensitivity_mean`. Use `val_balanced_accuracy_mean` as the tie-breaker and check `val_sensitivity_std`.
3. Run the Stage 2 notebook with the winning settings.

Do not choose the winner by the test columns.
